In [1]:
import pandas as pd
import sqlite3

In [2]:
# Creating database connection
conn = sqlite3.connect("../database/RoadAccident.db")

In [3]:
# Checking tables in the database
tables = pd.read_sql_query("select name from sqlite_master where type='table'", conn)
tables

,name
0,RoadAccidentSQL


In [4]:
# Iterates through all database tables to display 
# their record count and a preview of the first few rows 
# for data validation.

for table in tables["name"]:
    print('-' * 50, table, '-' * 50)

    count_df = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table}",
        conn
    )
    print("Count of records:", count_df["count"].iloc[0])

    display(
        pd.read_sql(
            f"SELECT * FROM {table} LIMIT 5",
            conn
        )
    )


-------------------------------------------------- RoadAccidentSQL --------------------------------------------------
Count of records: 307973


,accident_index,accident_date,day_of_week,junction_control,junction_detail,accident_severity,light_conditions,local_authority,carriageway_hazards,number_of_casualties,number_of_vehicles,police_force,road_surface_conditions,road_type,speed_limit,time,urban_or_rural_area,weather_conditions,vehicle_type
0,BS0000001,01-01-2021,Thursday,Give way or uncontrolled,T or staggered junction,Serious,Daylight,Kensington and Chelsea,None,1,2,Metropolitan Police,Dry,One way street,30,15:11,Urban,Fine no high winds,Car
1,BS0000002,05-01-2021,Monday,Give way or uncontrolled,Crossroads,Serious,Daylight,Kensington and Chelsea,None,11,2,Metropolitan Police,Wet or damp,Single carriageway,30,10:59,Urban,Fine no high winds,Taxi/Private hire car
2,BS0000003,04-01-2021,Sunday,Give way or uncontrolled,T or staggered junction,Slight,Daylight,Kensington and Chelsea,None,1,2,Metropolitan Police,Dry,Single carriageway,30,14:19,Urban,Fine no high winds,Taxi/Private hire car
3,BS0000004,05-01-2021,Monday,Auto traffic signal,T or staggered junction,Serious,Daylight,Kensington and Chelsea,None,1,2,Metropolitan Police,Frost or ice,Single carriageway,30,08:10,Urban,Other,Motorcycle over 500cc
4,BS0000005,06-01-2021,Tuesday,Auto traffic signal,Crossroads,Serious,Darkness - lights lit,Kensington and Chelsea,None,1,2,Metropolitan Police,Dry,Single carriageway,30,17:25,Urban,Fine no high winds,Car


## KPI´S

In [5]:
# Calculating the total number of casualties from the RoadAccidentSQL table

Total_Casualties = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Casualties
                                  FROM RoadAccidentSQL
                                  """, conn)
Total_Casualties

,CY_Casualties
0,417883


In [6]:
Total_Casualties_2022 = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Casualties_2021
    FROM RoadAccidentSQL
    WHERE substr(accident_date, -4) = '2021'
""", conn)

Total_Casualties_2022

,CY_Casualties_2021
0,222146


In [7]:
Total_Casualties_2022 = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Casualties_2022
    FROM RoadAccidentSQL
    WHERE substr(accident_date, -4) = '2022'
""", conn)

Total_Casualties_2022

,CY_Casualties_2022
0,195737


In [8]:
# Calculating the total number of casualties from the RoadAccidentSQL table where the accident severity is 'Fatal'

Fatal_Casualities = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Fatal_Casualties
                                  FROM RoadAccidentSQL
                                WHERE accident_severity = 'Fatal' 
                                  """, conn)
Fatal_Casualities

,CY_Fatal_Casualties
0,7135


In [9]:
# Calculating the total number of casualties from the RoadAccidentSQL table where the accident severity is 'Serious'

Serious_Casualities = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Serious_Casualties
                                  FROM RoadAccidentSQL
                                WHERE accident_severity = 'Serious' 
                                  """, conn)
Serious_Casualities

,CY_Serious_Casualties
0,59312


In [10]:
# Calculating the total number of casualties from the RoadAccidentSQL table where the accident severity is 'Slight'

Slight_Casualities = pd.read_sql_query("""
    SELECT SUM(number_of_casualties) AS CY_Slight_Casualties
                                  FROM RoadAccidentSQL
                                WHERE accident_severity = 'Slight' 
                                  """, conn)
Slight_Casualities

,CY_Slight_Casualties
0,351436


## Accidents per type cars

In [12]:
Accidents_per_type_cars = pd.read_sql_query("""
SELECT
    CASE
        WHEN vehicle_type = 'Agricultural vehicle' THEN 'Agricultural'

        WHEN vehicle_type IN ('Car', 'Taxi/Private hire car') 
            THEN 'Car'

        WHEN vehicle_type IN (
            'Motorcycle 125cc and under',
            'Motorcycle 50cc and under',
            'Motorcycle over 125cc and up to 500cc',
            'Motorcycle over 500cc',
            'Pedal cycle'
        ) THEN 'Bike'

        WHEN vehicle_type IN (
            'Bus or coach (17 or more pass seats)',
            'Minibus (8 - 16 passenger seats)'
        ) THEN 'Bus'

        WHEN vehicle_type IN (
            'Goods 7.5 tonnes mgw and over',
            'Goods over 3.5t and under 7.5t',
            'Van / Goods 3.5 tonnes mgw or under'
        ) THEN 'Van'

        WHEN vehicle_type IN ('Other vehicle', 'Ridden horse') 
            THEN 'Other'

        ELSE 'Other'
    END AS vehicle_group,
    COUNT(*) AS accidents
FROM RoadAccidentSQL
GROUP BY vehicle_group
""", conn)

Accidents_per_type_cars

,vehicle_group,accidents
0,Agricultural,749
1,Bike,25132
2,Bus,9507
3,Car,245337
4,Other,5021
5,Van,22227


In [21]:
Casualties_Monthly_Trend = pd.read_sql_query("""
    SELECT 
        CASE substr(accident_date, 4, 2)
            WHEN '01' THEN 'January'
            WHEN '02' THEN 'February'
            WHEN '03' THEN 'March'
            WHEN '04' THEN 'April'
            WHEN '05' THEN 'May'
            WHEN '06' THEN 'June'
            WHEN '07' THEN 'July'
            WHEN '08' THEN 'August'
            WHEN '09' THEN 'September'
            WHEN '10' THEN 'October'
            WHEN '11' THEN 'November'
            WHEN '12' THEN 'December'
        END AS Month_Name,
        CAST(substr(accident_date, 4, 2) AS INTEGER) AS Month_Num,
        SUM(number_of_casualties) AS 'CY Casualties'
    FROM RoadAccidentSQL
    WHERE substr(accident_date, 7, 4) = '2022'
    GROUP BY substr(accident_date, 4, 2)
    ORDER BY Month_Num
""", conn)

Casualties_Monthly_Trend = Casualties_Monthly_Trend.drop(columns=['Month_Num'])
Casualties_Monthly_Trend

,Month_Name,CY Casualties
0,January,13163
1,February,14804
2,March,16575
3,April,15767
4,May,16775
5,June,17230
6,July,17201
7,August,16796
8,September,17500
9,October,18287


In [18]:
# Step 1: Check if table has any data at all
check = pd.read_sql_query("SELECT COUNT(*) as total FROM RoadAccidentSQL", conn)
print(check)

    total
0  307973


In [19]:
# Step 2: Check what accident_date values actually look like
sample = pd.read_sql_query("SELECT accident_date FROM RoadAccidentSQL LIMIT 5", conn)
print(sample)

  accident_date
0    01-01-2021
1    05-01-2021
2    04-01-2021
3    05-01-2021
4    06-01-2021


In [20]:
# Check what the date values actually look like
sample = pd.read_sql_query("SELECT accident_date FROM RoadAccidentSQL LIMIT 10", conn)
print(sample)

  accident_date
0    01-01-2021
1    05-01-2021
2    04-01-2021
3    05-01-2021
4    06-01-2021
5    01-01-2021
6    08-01-2021
7    02-01-2021
8    07-01-2021
9    10-01-2021
